# team_refs

> Ideal free distribution, social optimum and team metrics (Phase 6).

Spec: `specs/multiagent/phase6/`. Reference allocations and ledger metrics for comparing teams to IFD and SO.

In [ ]:
#| default_exp team_refs


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations

import numpy as np


def yieldModel(world, p: int, n: int) -> float:
    "Total harvest rate of patch p with n agents (0 if n == 0)."
    if n <= 0:
        return 0.0
    if world.model == 'fixed':
        ip = world.intake[p]
        return min(n * world.c, ip) if np.isfinite(ip) else n * world.c
    msy = world.g * world.K / 4
    return n * world.c if n * world.c <= msy else world.g * world.Smin * (1 - world.Smin / world.K)


def idealFree(world, N: int):
    "Greedy IFD allocation over patches; returns (n per patch, total yield)."
    P = world.nPatches
    n = np.zeros(P, dtype=int)
    for _ in range(N):
        pay = [yieldModel(world, p, n[p] + 1) / (n[p] + 1) for p in range(P)]
        best = int(np.argmax(pay))
        if pay[best] <= 0:
            break
        n[best] += 1
    return n, float(sum(yieldModel(world, p, n[p]) for p in range(P)))


def socialOptimum(world, N: int):
    "Max total yield with at most N agents on patches (DP over patches)."
    P = world.nPatches
    best = np.zeros(N + 1)
    alloc = np.zeros((P, N + 1), dtype=int)
    for p in range(P):
        prev = best.copy()
        for m in range(N + 1):
            best[m], take = prev[m], 0
            for k in range(1, m + 1):
                y = prev[m - k] + yieldModel(world, p, k)
                if y > best[m]:
                    best[m], take = y, k
            alloc[p, m] = take
    m = int(np.argmax(best))
    n = np.zeros(P, dtype=int)
    rem = m
    for p in range(P - 1, -1, -1):
        n[p] = alloc[p, rem]
        rem -= n[p]
    return n, float(best[m])


def ledgerArrays(ledger):
    "Turn ledger rows into a dict of numpy arrays."
    if not ledger:
        spec = (('agent', int), ('t', float), ('dt', float), ('kind', object), ('location', int),
                ('patch', int), ('harvest', float), ('reward', float), ('s0', float), ('s1', float))
        return {k: np.array([], dtype=d) for k, d in spec}
    cols = list(zip(*ledger))
    return dict(agent=np.asarray(cols[0], int), t=np.asarray(cols[1], float), dt=np.asarray(cols[2], float),
                kind=np.asarray(cols[3], object), location=np.asarray(cols[4], int),
                patch=np.asarray([x if x is not None else -1 for x in cols[5]], int),
                harvest=np.asarray(cols[6], float), reward=np.asarray(cols[7], float),
                s0=np.asarray(cols[8], float), s1=np.asarray(cols[9], float))


def welfare(L, nAgents: int, Ttot: float):
    tot = float(L['harvest'].sum()) if len(L['harvest']) else 0.0
    return tot, tot / (nAgents * Ttot) if nAgents * Ttot > 0 else 0.0


def gini(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0 or x.sum() == 0:
        return 0.0
    d = np.abs(x[:, None] - x[None, :]).sum()
    return float(d / (2 * x.size ** 2 * x.mean()))


def occupancy(L, nAgents: int, P: int, Ttot: float, step: float = 1.0):
    "Agents per patch on a time grid; last column = outside (travelling or at a non-patch location)."
    times = np.arange(0.0, Ttot, step)
    counts = np.zeros((len(times), P + 1))
    for kind, patch, s0, s1 in zip(L['kind'], L['patch'], L['s0'], L['s1']):
        if kind != 'stay' or patch < 0 or not np.isfinite(s0):
            continue
        m = (times >= s0) & (times < s1)
        counts[m, patch] += 1
    counts[:, P] = nAgents - counts[:, :P].sum(axis=1)
    return times, counts


def coverage(counts):
    occupied = counts[:, :-1] > 0
    return float(occupied.mean(axis=1).mean()) if counts.size else 0.0


def dangerShare(L, danger, nAgents: int, Ttot: float):
    if nAgents * Ttot <= 0:
        return 0.0
    t = 0.0
    for kind, loc, dt in zip(L['kind'], L['location'], L['dt']):
        if kind == 'stay' and int(loc) in danger:
            t += float(dt)
    return t / (nAgents * Ttot)


def ifdDistance(counts, alloc, nAgents: int):
    target = np.zeros(counts.shape[1])
    target[: len(alloc)] = alloc
    target[-1] = nAgents - int(alloc.sum())
    return 0.5 * np.abs(counts - target).sum(axis=1) / nAgents


def timeToIFD(times, d, moveTimes, delta=0.2, w=10):
    "Per move: seconds until the mean of d over w grid points after the move first drops to delta (nan if not before the next move)."
    out = []
    ends = list(moveTimes[1:]) + [np.inf]
    for mt, end in zip(moveTimes, ends):
        idx = (times >= mt) & (times < end)
        seg, tseg = d[idx], times[idx]
        found = float('nan')
        for i in range(w - 1, len(seg)):
            if seg[i - w + 1:i + 1].mean() <= delta:
                found = float(tseg[i] - mt)
                break
        out.append(found)
    return out


def welfareLoss(total, soYield: float, Ttot: float):
    cap = soYield * Ttot
    return 1.0 - total / cap if cap > 0 else 0.0


## Tests

In [ ]:
from explRL.patchworld import TeamPatchWorld, ringLayout
from fastcore.test import test_eq, test_close
import numpy as np


In [ ]:
from fastcore.test import test_eq, test_close

# AC-7 references
w = TeamPatchWorld(ringLayout(8), nPatches=3, nDanger=0, model='fixed', intake=(1.0, 2.0, 3.0),
                   patchSites=[0, 1, 2], c=100.0, Tmove=None)
n, y = idealFree(w, 6)
test_eq(n.tolist(), [1, 2, 3])
test_close(y, 6.0)

rng = np.random.default_rng(0)

def brute_so(world, N):
    P = world.nPatches
    best_y, best_n = -1.0, None
    from itertools import product
    # enumerate allocations with sum <= N
    def gen(rem, p, cur):
        if p == P:
            yield tuple(cur)
            return
        for k in range(rem + 1):
            cur.append(k)
            yield from gen(rem - k, p + 1, cur)
            cur.pop()
    for alloc in gen(N, 0, []):
        if sum(alloc) > N:
            continue
        yy = sum(yieldModel(world, p, alloc[p]) for p in range(P))
        if yy > best_y:
            best_y, best_n = yy, alloc
    return np.array(best_n, int), best_y

for _ in range(50):
    P = int(rng.integers(2, 5))
    intake = tuple(rng.uniform(0.5, 3.0, P))
    N = int(rng.integers(1, 8))
    ww = TeamPatchWorld(ringLayout(12), nPatches=P, nDanger=0, model='fixed', intake=intake,
                        patchSites=list(range(P)), c=1.5, Tmove=None)
    n_ifd, _ = idealFree(ww, N)
    if n_ifd.sum() < N:                                  # agents left outside: no patch pays them anything
        assert all(yieldModel(ww, q, n_ifd[q] + 1) / (n_ifd[q] + 1) <= 0 for q in range(P))
    for p in range(P):
        if n_ifd[p] == 0:
            continue
        own = yieldModel(ww, p, n_ifd[p]) / n_ifd[p]
        for q in range(P):
            if q == p:
                continue
            alt = yieldModel(ww, q, n_ifd[q] + 1) / (n_ifd[q] + 1)
            assert own >= alt - 1e-9
    n_so, y_so = socialOptimum(ww, N)
    bn, by = brute_so(ww, N)
    test_close(y_so, by)

wL = TeamPatchWorld(ringLayout(8), nPatches=4, nDanger=0, model='logistic', intake=(1.0,) * 4,
                    patchSites=[0, 1, 2, 3], K=100.0, g=0.2, Smin=1.0, c=1.0, Tmove=None)
cap = sum(int(wL.g * wL.K // (4 * wL.c)) for _ in range(wL.nPatches))
N = cap + 4
_, y_ifd = idealFree(wL, N)
_, y_so = socialOptimum(wL, N)
assert y_so > y_ifd

# AC-9 metrics
test_eq(gini(np.ones(4)), 0.0)
test_close(gini(np.array([0., 0., 0., 10.])), 3 / 4)    # (n - 1) / n
test_eq(ledgerArrays([])['harvest'].size, 0)

# hand-built ledger, 2 patches, 3 agents, 4 s: agents 0 and 1 share patch 0 during [0, 2); agent 2 stays at
# non-patch location 5 (danger) during [0, 1), travels, then stays at patch 1 during [2, 4)
nan = float('nan')
ledger = [(0, 0.0, 2.0, 'stay', 3, 0, 1.0, 1.0, 0.0, 2.0),
          (1, 0.0, 2.0, 'stay', 3, 0, 1.0, 1.0, 0.0, 2.0),
          (2, 0.0, 1.0, 'stay', 5, None, 0.0, -1.0, 0.0, 1.0),
          (2, 1.0, 1.0, 'travel', 7, None, 0.0, 0.0, nan, nan),
          (2, 2.0, 2.0, 'stay', 7, 1, 4.0, 4.0, 2.0, 4.0)]
L = ledgerArrays(ledger)
test_eq(welfare(L, 3, 4.0), (6.0, 0.5))
times, counts = occupancy(L, 3, 2, 4.0)
test_eq(counts, np.array([[2, 0, 1], [2, 0, 1], [0, 1, 2], [0, 1, 2]]))   # the danger stay counts as outside
test_close(coverage(counts), 0.5)
test_close(dangerShare(L, {5}, 3, 4.0), 1 / 12)
d = ifdDistance(counts, np.array([2, 1]), 3)                               # IFD target: (2, 1, outside 0)
test_close(d, [1 / 3, 1 / 3, 2 / 3, 2 / 3])
test_eq(ifdDistance(np.array([[2., 1., 0.]]), np.array([2, 1]), 3), [0.0])
t1, t2 = timeToIFD(times, d, [0.0, 2.0], delta=0.4, w=2)                 # full windows of points after the move
test_eq(t1, 1.0); assert np.isnan(t2)
test_eq(timeToIFD(times, d, [0.0], delta=0.5, w=3), [2.0])                # window mean (1/3 + 1/3 + 2/3) / 3 <= 0.5
test_close(welfareLoss(6.0, 3.0, 4.0), 0.5)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
